

#Initializing the functions:

In [0]:
%run ../../Configurations/Init_Scripts



#Defining source and destination paths and Schema:

In [0]:
dbutils.widgets.text("CatalogName", "cd_prod")
CatalogName = dbutils.widgets.get("CatalogName")
 
dbutils.widgets.text('ExternalLocation_raw',"/mntprod_raw")
ExternalLocation_raw = dbutils.widgets.get('ExternalLocation_raw')
 
dbutils.widgets.text('ExternalLocation_silver',"/mntprod_silver")
ExternalLocation_silver = dbutils.widgets.get('ExternalLocation_silver')

In [0]:
# Srcpath_File_SAP_BW = '/mnt/raw/SAP/BW/Customer/ZHCUSTOMR.csv'
#-------Srcpath_File_SAP_S4 = '/mnt/raw/SAP/S4/Customer/ZBWI05225_CUSTOMER_AGN_CON.csv'
Srcpath_File_SAP_S4 = ExternalLocation_raw+'/SAP/S4/Customer/ZBWI05225_CUSTOMER_AGN_CON.csv'
#-------Delta_Table_Path = '/mnt/silver/DIMCustomerMaster'
Delta_Table_Path = ExternalLocation_silver+'/DIMCustomerMaster'

dbutils.widgets.text('ConfigId','13')
ConfigId = dbutils.widgets.get('ConfigId')

dbutils.widgets.text('SourceTypeId','2')
SourceTypeId = dbutils.widgets.get('SourceTypeId')

CreatedBy = 'ADB_CustomerMaster'
DeviceTypeCd = 'SAP'
MessageTypeCd = 'CustomerMaster'

from pyspark.sql.types import StructType, StructField,StringType
# Schema_BW = StructType([

# StructField("AccountID", StringType(), False),
# StructField("AccountCountryAlpha2Cd", StringType(), False),
# StructField("AccountCityNm", StringType(), False),
# StructField("AccountPrimaryNm", StringType(), False),
# StructField("AccountSecondaryNm", StringType(), False),
# StructField("Name3", StringType(), False),
# StructField("AccountPrimaryPhoneNbr", StringType(), False),
# StructField("AccountPostalCd", StringType(), False),
# StructField("AcctStateProvinceCd", StringType(), False),
# StructField("AccountStreetAddressTxt", StringType(), False),
# StructField("AccountCity2Nm", StringType(), False),
# StructField("FaxNbr", StringType(), False),
# StructField("Language", StringType(), False),
# StructField("BIC_ZSPLTCUST", StringType(), False),
# StructField("CreatedBy", StringType(), False),
# StructField("CreatedOn", StringType(), False),
# StructField("BIC_ZSAPOBLK", StringType(), False),
# StructField("Del_Block", StringType(), False),
# StructField("Bill_Block", StringType(), False),
# StructField("AccountGroupCd", StringType(), False),
# StructField("AccountAddressId", StringType(), False),
# StructField("PostalGISCd", StringType(), False),
# StructField("TaxID", StringType(), False),
# StructField("BIC_ZCONSTYPE", StringType(), False),
# StructField("BIC_ZDEANUM", StringType(), False),
# StructField("Email", StringType(), False),
# StructField("VATNbr", StringType(), False),
# StructField("BIC_ZDCTFNAME", StringType(), False),
# StructField("BIC_ZDCTLNAME", StringType(), False),
# StructField("BIC_ZDCTMNAME", StringType(), False),
# StructField("BIC_ZDCTNMPFX", StringType(), False),
# StructField("BIC_ZDCTNMSFX", StringType(), False),
# StructField("BIC_ZDCTPFDSG", StringType(), False),
# StructField("BIC_ZREASON", StringType(), False),
# StructField("BIC_ZPROVSTAT", StringType(), False),
# StructField("BIC_Z_UPDAT", StringType(), False) ])

Schema_S4 = StructType([
StructField("AccountID", StringType(), False),
StructField("AccountCountryAlpha2Cd", StringType(), False),
StructField("AccountCityNm", StringType(), False),
StructField("AccountPrimaryNm", StringType(), False),
StructField("AccountSecondaryNm", StringType(), False),
StructField("Name3", StringType(), False),
StructField("AccountPrimaryPhoneNbr", StringType(), False),
StructField("AccountPostalCd", StringType(), False),
StructField("AcctStateProvinceCd", StringType(), False),
StructField("AccountStreetAddressTxt", StringType(), False),
StructField("AccountCity2Nm", StringType(), False),
StructField("FaxNbr", StringType(), False),
StructField("Language", StringType(), False),
StructField("CreatedBy", StringType(), False),
StructField("CreatedOn", StringType(), False),
StructField("BIC_ZSAPOBLK", StringType(), False),
StructField("Del_Block", StringType(), False),
StructField("Bill_Block", StringType(), False),
StructField("AccountGroupCd", StringType(), False),
StructField("AccountAddressId", StringType(), False),
StructField("PostalGISCd", StringType(), False),
StructField("TaxID", StringType(), False),
StructField("_BIC_ZCONSTYPE", StringType(), False),
StructField("Email", StringType(), False),
StructField("VATNbr", StringType(), False) ])

# Data Ingestion

In [0]:
#Widgets
dbutils.widgets.text('IngestionConfigId','6')
IngestionConfigId = dbutils.widgets.get('IngestionConfigId')

job_id=dbutils.widgets.text("job_id","-1")
job_id=dbutils.widgets.get("job_id")

run_id=dbutils.widgets.text("run_id","-1")
run_id=dbutils.widgets.get("run_id")

source_filename=dbutils.widgets.text("source_filename","ZBWI05225_CUSTOMER_AGN_CON.csv")
source_filename=dbutils.widgets.get("source_filename")

source_filename_list = source_filename.split(',')
print("Source File Name to Process:"+str(source_filename_list))

#-----------src_filepath = '/mnt/bwsourcedata/'
src_filepath = ExternalLocation_raw+'/bwsourcedata/'

In [0]:
for src_filename in source_filename_list:
    if src_filename.startswith('ZBWI05225_CUSTOMER'):
        dest_filepath = Srcpath_File_SAP_S4.replace(src_filename,"")
    elif src_filename.startswith('ZHCUSTOM'):
        dest_filepath = Srcpath_File_SAP_BW.replace(src_filename,"")
        
    src_path = src_filepath+src_filename
    SourceContainerPath = src_filepath.split('/')[2]
    SourceFolderPath = src_filepath.split('/')[2]
    DestinationContainerPath = dest_filepath.split('/')[2]
    DestinationFolderPath = dest_filepath[dest_filepath.find('/',1)+1:dest_filepath.rfind('/',1)]
    try:
        for fileproperty in dbutils.fs.ls(src_path): filesize = fileproperty.size
        dbutils.fs.cp(src_path,dest_filepath)

        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':SourceContainerPath,'SourceFolderPath':SourceFolderPath,'SourceFileName':src_filename,
                        'DestinationContainerPath':DestinationContainerPath,'DestinationFolderPath':DestinationFolderPath,'DestinationFileName':src_filename,'PipelineStatus':"Succeeded",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd,'SourceFileSize':filesize
                        }]) 

        logIntoIngestionLogTable(processedFileList,'LoadFiles_CustomerMaster') 
    except Exception as e:
        processedFileList =([{'ConfigId':IngestionConfigId,'SourceTypeId':SourceTypeId,'SourceContainerPath':SourceContainerPath,'SourceFolderPath':SourceFolderPath,'SourceFileName':src_filename,
                        'DestinationContainerPath':DestinationContainerPath,'DestinationFolderPath':DestinationFolderPath,'DestinationFileName':src_filename,'PipelineStatus':"Failed",'PipelineRunId':run_id, 'JobId':job_id,'DeviceType':DeviceTypeCd,'LogType':MessageTypeCd,'ErrorMessage':str(e)
                        }]) 
        logIntoIngestionLogTable(processedFileList,'LoadFiles_CustomerMaster') 
        raise e



#Reading Raw and combined with common cols BW and S4 data

In [0]:
## Combined New and Old customer master files, Write into silver zone Customer delta table

cols_BW_S4 =["AccountID","AccountPrimaryNm","AccountSecondaryNm","AccountGroupCd","AccountAddressId","AccountCityNm",
    "AccountCountryAlpha2Cd","AccountPrimaryPhoneNbr","AccountPostalCd","AcctStateProvinceCd","AccountStreetAddressTxt",
    "AccountCity2Nm","CreatedBy", "CreatedDt", "UpdatedBy", "UpdatedDt","SourceFilePath","SourceFileName","SourceFileSize","FileExtractedTimstmp"]

# df_raw_BW = spark.read.schema(Schema_BW).options(delimiter='|').csv(Srcpath_File_SAP_BW)\
# .select('*',col('_metadata.file_path').alias('SourceFilePath')\
#            ,col("_metadata.file_name").alias('SourceFileName')\
#            ,col("_metadata.file_modification_time").alias('FileExtractedTimstmp')\
#            ,col("_metadata.file_size").alias('SourceFileSize'))\
# .withColumn('SourceFilePath', regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''))\
# .withColumn("CreatedBy",lit("ADB_CustomerMaster"))\
# .withColumn("CreatedDt",current_timestamp())\
# .withColumn("UpdatedBy",lit("ADB_CustomerMaster"))\
# .withColumn("UpdatedDt",current_timestamp())\
# .select(cols_BW_S4)

df_raw_S4 = spark.read.schema(Schema_S4).options(delimiter='|').csv(Srcpath_File_SAP_S4)\
.select('*',col('_metadata.file_path').alias('SourceFilePath')\
           ,col("_metadata.file_name").alias('SourceFileName')\
           ,col("_metadata.file_modification_time").alias('FileExtractedTimstmp')\
           ,col("_metadata.file_size").alias('SourceFileSize'))\
.withColumn('SourceFilePath', regexp_replace('SourceFilePath','dbfs:/mnt/raw/',''))\
.withColumn("CreatedBy",lit("ADB_CustomerMastewr"))\
.withColumn("CreatedDt",current_timestamp())\
.withColumn("UpdatedBy",lit("ADB_CustomerMastewr"))\
.withColumn("UpdatedDt",current_timestamp())\
.select(cols_BW_S4)

# Combined both BW and S4 and filter with the AccountPrimaryNm is not null
# df_union_RAW_BW_S4 = df_raw_BW.union(df_raw_S4).dropDuplicates()
df_raw_S4 = df_raw_S4.dropDuplicates()
df_union_RAW_BW_S4 = df_raw_S4.filter(col("AccountPrimaryNm").isNotNull())

df_union_RAW_BW_S4 = (df_union_RAW_BW_S4
.withColumn('ConfigId', lit(ConfigId).cast('int'))
.withColumn('SourceTypeId', lit(SourceTypeId).cast('int'))
.withColumn("DeviceType",lit(DeviceTypeCd))
.withColumn("LogType",lit(MessageTypeCd))
.withColumn("DeviceId",lit(None).cast('string')))

In [0]:
# Read logsourcefileprocessed from silver zone:
logsourcefileprocessed_df = (spark.read.format('delta')
                             .option('header', True)
                             .load(src_filesProcessed)
                             .filter("LogFileStatus = 'InProgress'")
                             .select('SourceFilePath','FileNameUUID'))  



#Merge on delta table

In [0]:
try:
    # loadAuditTables_DIM_InProgress(df_union_RAW_BW_S4,Delta_Table_Path,13,2,'ADB_CustomerMaster')

    loadAuditTables_Ingestion_Log(df_union_RAW_BW_S4,Delta_Table_Path,CreatedBy,'InProgress','')

    df_union_RAW_BW_S41 = (df_union_RAW_BW_S4.groupBy("SourceFilePath","SourceFileName","SourceFileSize","ConfigId","SourceTypeId",'DeviceId',"FileExtractedTimstmp").count()
                          .withColumnRenamed("count",'RecdCnt')
                          .withColumn("FileNameDeviceTypeCd",lit(DeviceTypeCd))
                          .withColumn("ExternalSerialNbr",lit(None))
                          .withColumn("InternalSerialNbr",lit(None))
                          .withColumn("FileNameMessageTypeCd",lit(MessageTypeCd))
                          .withColumn("FileNameDtTmstmp",date_format('FileExtractedTimstmp', 'yyyyMMddHHmmss'))
                          .withColumn("FileNameApplicatorPortCd",lit(None))
                          .withColumn("FileNameCycleNbr",lit(None))
                          .withColumn('LogStartDate',lit(None))
                          .withColumn('LogEndDate',lit(None)))

    loadlogProcessesDeltaTable(df_union_RAW_BW_S41,Delta_Table_Path,CreatedBy,'InProgress','')

    # Taking latest accountid
    w=Window.partitionBy("accountid").orderBy(desc("accountid"),desc("SourceFileName"),desc("accountcitynm"),desc("accountcity2nm"))
    dedupeDF_RAW_BW_S4 = (df_union_RAW_BW_S4.drop_duplicates()
                                            .withColumn("row_Num",row_number().over(w)).filter("row_num = 1").drop('row_num')
                         )
    #display(dedupeDF_RAW_BW_S4)
    # Joining with logfileprocessed table to get FilenameUUID
    dedupeDF_RAW_BW_S4_UUID = dedupeDF_RAW_BW_S4.join(logsourcefileprocessed_df,"SourceFilePath",'inner').drop_duplicates()

    DeltaTbl_Target = DeltaTable.forPath(spark, Delta_Table_Path)  
    DeltaTbl_Target.alias("tgt") \
            .merge(
            dedupeDF_RAW_BW_S4_UUID.alias("src"),
                "src.AccountID = tgt.AccountID") \
     .whenMatchedUpdate(set =
      {
        "tgt.FileNameUUID": "src.FileNameUUID",        
        "tgt.AccountPrimaryNm": "src.AccountPrimaryNm",
        "tgt.AccountSecondaryNm": "src.AccountSecondaryNm",
        "tgt.AccountGroupCd": "src.AccountGroupCd",
        "tgt.AccountAddressId": "src.AccountAddressId",
        "tgt.AccountCityNm": "src.AccountCityNm",
        "tgt.AccountCity2Nm": "src.AccountCity2Nm",          
        "tgt.AccountCountryAlpha2Cd": "src.AccountCountryAlpha2Cd",
        "tgt.AccountPrimaryPhoneNbr": "src.AccountPrimaryPhoneNbr",
        "tgt.AccountPostalCd": "src.AccountPostalCd",
        "tgt.AcctStateProvinceCd": "src.AcctStateProvinceCd",
        "tgt.AccountStreetAddressTxt": "src.AccountStreetAddressTxt",
        "tgt.ProcessedByNm": "src.SourceFileName",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDt": "src.UpdatedDt"
      }
    ) \
     .whenNotMatchedInsert(values =
      {
        "tgt.FileNameUUID": "src.FileNameUUID",
        "tgt.AccountID": "src.AccountID",
        "tgt.AccountPrimaryNm": "src.AccountPrimaryNm",
        "tgt.AccountSecondaryNm": "src.AccountSecondaryNm",
        "tgt.AccountGroupCd": "src.AccountGroupCd",
        "tgt.AccountAddressId": "src.AccountAddressId",
        "tgt.AccountCityNm": "src.AccountCityNm",
        "tgt.AccountCity2Nm": "src.AccountCity2Nm",          
        "tgt.AccountCountryAlpha2Cd": "src.AccountCountryAlpha2Cd",
        "tgt.AccountPrimaryPhoneNbr": "src.AccountPrimaryPhoneNbr",
        "tgt.AccountPostalCd": "src.AccountPostalCd",
        "tgt.AcctStateProvinceCd": "src.AcctStateProvinceCd",
        "tgt.AccountStreetAddressTxt": "src.AccountStreetAddressTxt",
        "tgt.ProcessedByNm": "src.SourceFileName",                    
        "tgt.CreatedBy": "src.CreatedBy",
        "tgt.CreatedDt": "src.CreatedDt",
        "tgt.UpdatedBy": "src.UpdatedBy",
        "tgt.UpdatedDt": "src.UpdatedDt"
      }
    ) \
      .execute()
    # loadAuditTables_DIM_Complete(df_union_RAW_BW_S4,Delta_Table_Path,13,2,'ADB_CustomerMaster','Succeeded','')
    dedupeDF_RAW_BW_S41 = (dedupeDF_RAW_BW_S4_UUID.withColumn('ConfigId',lit(ConfigId)).groupBy('SourceFilePath','SourceFileName','FileNameUUID','ConfigId')
                                    .agg(min(col('FileExtractedTimstmp')).alias('LogStartDate'),
                                         max(col('FileExtractedTimstmp')).alias('LogEndDate')))
    
    loadlogProcessesDeltaTable(dedupeDF_RAW_BW_S41,Delta_Table_Path,CreatedBy,'Succeeded','')
    loadAuditTables_Ingestion_Log(dedupeDF_RAW_BW_S4,Delta_Table_Path,CreatedBy,'Succeeded','')
except Exception as e:
    print(str(e))
    # loadAuditTables_DIM_Complete(df_union_RAW_BW_S4,Delta_Table_Path,13,2,'ADB_CustomerMaster','Failed',str(e))
    loadlogProcessesDeltaTable(dedupeDF_RAW_BW_S41,Delta_Table_Path,CreatedBy,'Failed',str(e))
    loadAuditTables_Ingestion_Log(dedupeDF_RAW_BW_S4,Delta_Table_Path,CreatedBy,'Failed',str(e))
    raise    